<a href="https://colab.research.google.com/github/NikosFilippov/tonights-front-desk/blob/main/%CE%91%CE%BD%CF%84%CE%AF%CE%B3%CF%81%CE%B1%CF%86%CE%BF_M1_10_assignment_study_office.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎓 Semester at the study office · starter notebook

**The situation.** About one first-year student in five leaves the university during the first semester. Many of
them could have been helped: a conversation with an adviser, a study group, help with money. But the study office
has only **three advisers**. At the end of **week 6** they can hold about **40 conversations**.

**Your job:** build a model that helps the office decide *who* to talk to, find out what **mistakes** it makes
(students worried for nothing, students missed), turn it into a **rule** the office can use, and then build a small
**Streamlit app** for the office.

**How to work.** Go through the steps in order. Each ✏️ cell is yours to write. Use the hotel notebook (session 10) as your template: it
does every step for hotel bookings. AI assistants are welcome for the code, but make sure you understand each step:
you will explain it in your video. `GUIDE.md` explains the terms on one page.

*The students are synthetic, but every profile is resampled from a real student (UCI "Predict students' dropout
and academic success", 4,424 students, CC BY 4.0), and the risk of leaving follows a model fitted on the real
outcomes.*

In [28]:
%pip install -q optuna shap
import json, time, warnings, platform
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import sklearn, xgboost as xgb, optuna, shap
warnings.filterwarnings("ignore", category=UserWarning)
optuna.logging.set_verbosity(optuna.logging.WARNING)
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingRegressor, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

In [29]:
import os
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, confusion_matrix
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import xgboost as xgb

# the data lives on GitHub, so this works in Colab without uploading anything
URL = "https://raw.githubusercontent.com/aaubs/ds-master/main/assignments/study-office/data/"
BASE = "data/" if os.path.exists("data/history_week6.csv") else URL
history = pd.read_csv(BASE + "history_week6.csv")   # past students, with what happened
new = pd.read_csv(BASE + "new_week6.csv")           # this year's students, week 6: what happens is not known yet
print(history.shape, new.shape)
history.head()

(2686, 24) (703, 20)


,student_id,cohort,age,admission_grade,international,first_gen,su_scholarship,fees_owed,moved_from_home,married,...,logins_last3,logins_trend,submitted_share,missed_last3,quiz_mean,weeks_since_login,ects_passed_sem1,deregistration_form_opened,last_login_week,left
0,S230000,2023,19,9.1,0,1,0,0,1,0,...,23.0,-3.67,1.00,0.0,70.83,0.0,6,0,12,0
1,S230001,2023,35,5.2,0,0,0,0,0,0,...,16.0,1.00,0.83,0.0,55.20,0.0,5,0,12,0
2,S230002,2023,18,6.9,0,1,0,0,0,0,...,15.0,-1.67,1.00,0.0,67.33,0.0,8,0,12,0
3,S230003,2023,21,7.2,0,1,0,0,0,0,...,7.0,-1.67,0.33,2.0,70.00,0.0,5,0,12,0
4,S230004,2023,21,7.1,0,1,0,0,1,0,...,15.0,-1.33,0.67,0.0,66.75,0.0,3,0,12,0


---
## 0 · Think first 🤔

Before any code, write down in your own words:

1. **Who decides what?** Who uses the model's output, and what exactly do they decide?
2. **When?** What does the office know about a student at the end of week 6, and what does it not know yet?
3. **The two mistakes.** The model can flag a student who would have stayed anyway (a **false alarm**, a *false
   positive*), or miss a student who then leaves (a **missed student**, a *false negative*). What does each cost,
   and for whom?
4. **Which is worse?** Is one false alarm as bad as one missed student?

*Your answers:*

1. The model's output is used by Aalborg University offices to determine which students would need a bit of a check up on their studies in order to reduce the droping out of students
2.
3. For the "false alarms" the officials are losing a counsiling session with a student that was on the verge of dropping out so we need to figure out a way to calculate efficiently both because our student sessions are a handfull
4. I believe the false negative is substantialy worse since after a student has already left there is not a case of returning to studies while the false positive event could be a wake up call for a potential drop- off

---
## 1 · Look at the data

**`history_week6.csv`**: one row per student from the 2023, 2024 and 2025 cohorts who was still enrolled at the end
of week 6, with what the office knew then, and **`left`** (1 = left later that semester). **`new_week6.csv`**: the
2026 students, this week. This is the list the office has to act on.

| Column | Meaning |
|---|---|
| `age`, `gender`, `programme`, `evening_programme` | from enrolment |
| `admission_grade` | grade point average from upper secondary school (Danish scale) |
| `international`, `first_gen` | international student; first in the family at university |
| `su_scholarship`, `fees_owed`, `moved_from_home`, `married` | money and life situation (0/1) |
| `logins_total`, `logins_last3`, `logins_trend` | logins on the learning platform: weeks 1–6, the last three weeks, and the change |
| `submitted_share`, `missed_last3` | share of weekly assignments handed in; assignments missed in the last three weeks |
| `quiz_mean`, `weeks_since_login` | average quiz score; weeks since the last login |
| `ects_passed_sem1`, `deregistration_form_opened`, `last_login_week` | ⚠️ see step 2 |
| `left` | **the target**: 1 = left later in the semester |

✏️ **Your turn.** How many students leave? Compare the share who left across a few columns you think matter, for example `programme`, `fees_owed`, `submitted_share`. (Template: section 1 of the hotel notebook (session 10).)

<details><summary>Hint</summary>

`history['left'].mean()`; `history.groupby('fees_owed')['left'].mean()`; for a number, cut it into groups first: `history.groupby(pd.cut(history['submitted_share'], [0, .5, .8, 1], include_lowest=True))['left'].mean()`.
</details>

In [30]:
print("about " + str(history['left'].mean()) + " of university students leave(percentage)")
print(history.groupby('fees_owed')['left'].mean())
print(history.groupby(pd.cut(history['submitted_share'], [0, .5, .8, 1], include_lowest=True))['left'].mean())


about 0.08153387937453463 of university students leave(percentage)
fees_owed
0    0.067397
1    0.189103
Name: left, dtype: float64
submitted_share
(-0.001, 0.5]    0.212714
(0.5, 0.8]       0.097760
(0.8, 1.0]       0.047032
Name: left, dtype: float64


/tmp/ipykernel_3462/813528787.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(history.groupby(pd.cut(history['submitted_share'], [0, .5, .8, 1], include_lowest=True))['left'].mean())


So about 8.1% of students are leaving and of those the ones that have fees_owned are about 19% in comparisson to 6.7% of those that are staying so one strong indicator is if the fees are owned. we also notice that the submission share for each quentile (0, 0.5,0.8,1). Furthermore we will include the describtion for each data in the collumn to check for any prices that wither dont make sense logically or are too extreme to be true. i will also groupby the results by people left and people not leaving to know the average and std of the each population we are studuying.

In [31]:
for i in history.columns:
  print(i)
  print(history.groupby("left")[i].describe())

student_id
     count unique      top freq
left                           
0     2467   2467  S251049    1
1      219    219  S230013    1
cohort
       count         mean       std     min     25%     50%     75%     max
left                                                                       
0     2467.0  2024.036887  0.817897  2023.0  2023.0  2024.0  2025.0  2025.0
1      219.0  2023.981735  0.783801  2023.0  2023.0  2024.0  2025.0  2025.0
age
       count       mean       std   min   25%   50%   75%   max
left                                                           
0     2467.0  22.417916  6.755915  18.0  18.0  20.0  23.0  60.0
1      219.0  26.315068  9.389449  18.0  20.0  23.0  30.0  60.0
admission_grade
       count      mean       std  min  25%  50%  75%   max
left                                                      
0     2467.0  7.426794  2.137718  2.0  6.0  7.4  8.9  12.0
1      219.0  6.936073  2.420275  2.0  5.3  6.9  8.7  12.0
international
       count      mean  

In [32]:
history.isna().sum()

,0
student_id,0
cohort,0
age,0
admission_grade,0
international,0
first_gen,0
su_scholarship,0
fees_owed,0
moved_from_home,0
married,0


Also we notice there is a bit of missing data on the quiz_mean after checking it out we notice its from both students that left and not so its probably by students unable to take the quiz. Therefore we will fill it up with 0

In [33]:
history[history["quiz_mean"].isna()]

,student_id,cohort,age,admission_grade,international,first_gen,su_scholarship,fees_owed,moved_from_home,married,...,logins_last3,logins_trend,submitted_share,missed_last3,quiz_mean,weeks_since_login,ects_passed_sem1,deregistration_form_opened,last_login_week,left
48,S230055,2023,21,5.6,0,1,0,0,0,0,...,9.0,1.00,0.0,3.0,NaN,0.0,3,0,11,0
178,S230200,2023,32,6.4,0,1,0,0,0,0,...,6.0,1.33,0.0,3.0,NaN,0.0,0,1,11,1
593,S230651,2023,28,4.5,0,1,0,0,0,1,...,8.0,0.67,0.0,3.0,NaN,0.0,0,1,6,1
694,S230770,2023,25,5.4,0,1,1,0,0,1,...,5.0,-0.67,0.0,3.0,NaN,0.0,3,0,12,0
878,S240032,2024,19,6.8,0,1,0,0,1,0,...,4.0,0.00,0.0,3.0,NaN,0.0,3,0,12,0
1270,S240474,2024,20,6.1,0,1,0,0,1,0,...,3.0,-2.00,0.0,3.0,NaN,1.0,3,0,12,0
1296,S240502,2024,29,4.9,0,1,0,0,0,0,...,15.0,1.67,0.0,3.0,NaN,0.0,3,0,12,0
1329,S240539,2024,20,4.1,0,1,0,0,1,0,...,6.0,-1.00,0.0,3.0,NaN,0.0,0,1,10,1
1416,S240638,2024,20,8.4,0,1,1,0,1,0,...,5.0,-0.33,0.0,3.0,NaN,0.0,0,0,7,1
1456,S240681,2024,22,7.8,1,1,0,0,1,0,...,4.0,-1.33,0.0,3.0,NaN,0.0,6,0,12,0


In [34]:
history["quiz_mean"].fillna(0, inplace=True)

/tmp/ipykernel_3462/87063333.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  history["quiz_mean"].fillna(0, inplace=True)


We also notice a few trends checking the data like that. We notice that the average age for people leaving is a bit higher, that its more likely to not have a su scholarship , more likely to be married, fewer logins both in total and in the last 3 also in weeks since last login, lower submitted share of assignments, logically a higher deregistration form opened.

---
## 2 · What is known when? (leakage)

The model will be used **at the end of week 6**. A column that is only filled in later makes the model look
brilliant here and useless in October. Compare the two files: which columns does the history have that this week's
list does not?

In [35]:
sorted(set(history.columns) - set(new.columns))

['deregistration_form_opened', 'ects_passed_sem1', 'last_login_week', 'left']

✏️ **Your turn.** For each of these columns: when is it recorded? Would the office know it at the end of week 6? Write one sentence each, then make a list `features` of the columns the model may use (not `student_id`, `cohort`, the leaks, or the target). (Template: the leakage hunt, section 3 of the hotel notebook (session 10).)

<details><summary>Hint</summary>

ECTS points are awarded after the exams. A deregistration form is opened by students who are about to leave. The *last* login week is computed over the whole semester. `left` is the target itself.
</details>

#Answer
First of all we will not include the 'deregistration_form_opened', 'ects_passed_sem1', 'last_login_week' for the training since including this data can make our predictor depend heavily on those columns while they will not be accesible on the current data so our features will be everything apart from those. Also we will chacge the student_id into a string rather than not including it at all

*Your answer:*

In [36]:
history.dtypes

,0
student_id,object
cohort,int64
age,int64
admission_grade,float64
international,int64
first_gen,int64
su_scholarship,int64
fees_owed,int64
moved_from_home,int64
married,int64


In [37]:
history.columns.to_list()

['student_id',
 'cohort',
 'age',
 'admission_grade',
 'international',
 'first_gen',
 'su_scholarship',
 'fees_owed',
 'moved_from_home',
 'married',
 'evening_programme',
 'programme',
 'gender',
 'logins_total',
 'logins_last3',
 'logins_trend',
 'submitted_share',
 'missed_last3',
 'quiz_mean',
 'weeks_since_login',
 'ects_passed_sem1',
 'deregistration_form_opened',
 'last_login_week',
 'left']

In [38]:
history["student_id"] = history["student_id"].fillna(0).astype(str)

In [39]:
features = [
 'age',
 'admission_grade',
 'international',
 'first_gen',
 'su_scholarship',
 'fees_owed',
 'moved_from_home',
 'married',
 'evening_programme',
  'programme',
 'gender',
 'logins_total',
 'logins_last3',
 'logins_trend',
 'submitted_share',
 'missed_last3',
 'quiz_mean',
 'weeks_since_login'
 ]

---
## 3 · A split by time

The model will be used on **next year's** students, so the split follows time: learn from the 2023 and 2024
cohorts, check on 2025.

✏️ **Your turn.** Make `train` (2023 and 2024) and `val` (2025). How many students are in each, and what share leaves? (Template: section 2 of the hotel notebook (session 10).)

<details><summary>Hint</summary>

`train = history[history['cohort'] <= 2024]` and `val = history[history['cohort'] == 2025].copy()`.
</details>

In [40]:
train = history[history['cohort'] <= 2024]
val = history[history['cohort'] == 2025].copy()

for name, part in [("train", train), ("validation", val)]:
    print(f"{name:10s} {len(part):6,d} students · left {part['left'].mean():.1%}")

train       1,749 students · left 8.8%
validation    937 students · left 6.9%


---
## 4 · Train a model

Train two models on `train` and compare them on `val` with the **AUC**: pick one student who left and one who
stayed; the AUC is how often the model gives the one who left the higher risk (0.5 = coin flip, 1 = perfect).

1. A **logistic regression**: easy to read. Which features push the risk up or down?
2. **XGBoost**, as in the hotel notebook (session 10). It may not win here, and that is a result, not a failure.

Build both as a pipeline: first prepare the data (fill gaps, scale numbers, one-hot the categories), then the
model. Use the same preparation as the hotel notebook, so you can store the model the same way in step 9.

✏️ **Your turn.** Train the logistic regression, compute its validation AUC, and plot its coefficients.

<details><summary>Hint</summary>

```python
numeric = [c for c in features if c not in ["programme", "gender"]]
categorical = ["programme", "gender"]
prepare = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric),
    ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=20, sparse_output=False), categorical)])
logit = make_pipeline(prepare, LogisticRegression(max_iter=2000))
logit.fit(train[features], train["left"])
val["risk"] = logit.predict_proba(val[features])[:, 1]
```
Coefficients: `logit[-1].coef_[0]`, with names from `logit[0].get_feature_names_out()`.
</details>

In [41]:
numeric = [c for c in features if c not in ["programme", "gender"]]
categorical = ["programme", "gender"]
prepare = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric),
    ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=20, sparse_output=False), categorical)])
logit = make_pipeline(prepare, LogisticRegression(max_iter=2000))
logit.fit(train[features], train["left"])
val["risk"] = logit.predict_proba(val[features])[:, 1]

*Your answer:*

I want to test evert algorithm that we know of rather than just the logistic

In [42]:
def preprocessor(num, cat):
    # numbers: fill gaps, scale · categories: one column per level, rare levels pooled, unseen levels tolerated
    return ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num),
        ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=20, sparse_output=False), cat)])

def fit_and_score(model, num=numeric, cat=categorical):
    pipe = make_pipeline(preprocessor(num, cat), model).fit(train[num + cat], train["left"])
    p = pipe.predict_proba(val[num + cat])[:, 1]
    return pipe, p, roc_auc_score(val["left"], p)

booster = lambda: xgb.XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=3, subsample=0.8,
                                    colsample_bytree=0.8, tree_method="hist", random_state=0)

_, _, auc_clean = fit_and_score(booster())
print(f"Validation AUC: {auc_clean:.3f}")

ladder = {"baseline (always the average)": DummyClassifier(strategy="prior"),
          "logistic regression": LogisticRegression(max_iter=2000),
          "one deep tree": DecisionTreeClassifier(random_state=0),
          "random forest (bagging)": RandomForestClassifier(n_estimators=300, min_samples_leaf=2, n_jobs=-1, random_state=0),
          "XGBoost, default settings": xgb.XGBClassifier(tree_method="hist", random_state=0),
          "XGBoost, slower learning": booster()}

rows, val_proba = [], {}
for name, model in ladder.items():
    t0 = time.time()
    _, p, auc = fit_and_score(model)
    val_proba[name] = p
    rows.append((name, auc, log_loss(val["left"], np.clip(p, 1e-6, 1 - 1e-6)), time.time() - t0))

pd.DataFrame(rows, columns=["model", "AUC", "log loss", "seconds"]).set_index("model").round(3)

Validation AUC: 0.792


,AUC,log loss,seconds
model,,,
baseline (always the average),0.500,0.254,0.025
logistic regression,0.809,0.205,0.040
one deep tree,0.572,1.843,0.054
random forest (bagging),0.753,0.221,1.108
"XGBoost, default settings",0.764,0.274,1.314
"XGBoost, slower learning",0.792,0.214,0.554


#So we notice that logistic regression on our problem gives us about 81% accuracy while XGBoost gives us 79.2% while also noticing the always guessing the average or other methods. XGB and logistic are the 2 better options

✏️ **Your turn.** Train XGBoost the same way and compare the AUC. Pick one model and store its predictions for 2025 in `val['risk']`: all the next steps use it.

<details><summary>Hint</summary>

`make_pipeline(prepare, xgb.XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=3))`. If you want the app to be as easy as the hotel app, use XGBoost there: the hotel's way of storing the model is written for XGBoost.
</details>

In [43]:
# your code here

---
## 5 · The mistakes: the confusion matrix ⚠️

**This is the core of the assignment.** Suppose the office talks to every student with a risk of at least 50 %.
Every student lands in one of four boxes:

| | **not contacted** | **contacted** |
|---|---|---|
| **stayed** | ✅ true negative (TN) | 📞 **false positive (FP)**: a false alarm, a worried student who was fine |
| **left** | 🚪 **false negative (FN)**: a missed student, nobody tried | 🎯 true positive (TP): reached in time |

| Question | Name | Formula |
|---|---|---|
| Of the students we contacted, how many were really at risk? | **precision** | TP / (TP + FP) |
| Of the students who left, how many did we reach? | **recall** | TP / (TP + FN) |
| Of the students who were fine, how many did we worry? | **false positive rate** | FP / (FP + TN) |
| How many students did we handle correctly? | **accuracy** | (TP + TN) / all |

✏️ **Your turn.** Count the four boxes for a 50 % cut-off, compute the four numbers, and write each as a sentence about students (*"of the students we contacted, … were really at risk"*). Then compute the accuracy of contacting **nobody**. What does it tell you about accuracy? (Template: *Reading the confusion matrix*, section 7 of the hotel notebook (session 10).)

<details><summary>Hint</summary>

`TN, FP, FN, TP = confusion_matrix(val['left'], val['risk'] >= 0.5).ravel()`. Contacting nobody gets every student who stayed right, so its accuracy is `1 - val['left'].mean()`.
</details>

In [44]:
TN, FP, FN, TP = confusion_matrix(val['left'], val['risk'] >= 0.5).ravel()
print(TN,FP,FN,TP)

862 10 56 9


In [45]:
p_val_final = val["risk"].values
truth = val["left"].values

def boxes(t):
    called = p_val_final >= t
    return {"TP": int((called & (truth == 1)).sum()), "FP": int((called & (truth == 0)).sum()),
            "FN": int((~called & (truth == 1)).sum()), "TN": int((~called & (truth == 0)).sum())}
b = boxes(0.5)
pd.DataFrame([[b["TN"], b["FP"], b["TN"] + b["FP"]],
              [b["FN"], b["TP"], b["FN"] + b["TP"]],
              [b["TN"] + b["FN"], b["FP"] + b["TP"], len(truth)]],
             index=["stayed", "left", "total"], columns=["not contacted", "contacted", "total"])

,not contacted,contacted,total
stayed,862,10,872
left,56,9,65
total,918,19,937


In [46]:
precision = b["TP"] / (b["TP" ] + b["FP"])
recall = b["TP"] / (b["TP"] + b["FN"])
fpr = b["FP"] / (b["FP"] + b["TN"])
accuracy = (b["TP"] + b["TN"]) / len(truth)

print(f"Όριο κινδύνου (cut-off) 50%:")
print(f"  Επαφές (calls): {b['TP'] + b['FP']:,} φοιτητές")
print(f"  Precision {precision:.0%}: Από τους φοιτητές που καλέσαμε, το {precision:.0%} πραγματικά αποχώρησε.")
print(f"  Recall    {recall:.0%}: Εντοπίσαμε το {recall:.0%} των συνολικών αποχωρήσεων· οι υπόλοιποι χάθηκαν.")
print(f"  False Positive Rate {fpr:.1%}: Από τους φοιτητές που θα παρέμεναν, ανησυχήσαμε άδικα το {fpr:.1%}.")
print(f"  Accuracy  {accuracy:.1%}")
print(f"  Αν δεν καλούσαμε κανέναν, η ακρίβεια θα ήταν {1 - truth.mean():.1%} (αλλά θα χάναμε το 100% των αποχωρήσεων).")

Όριο κινδύνου (cut-off) 50%:
  Επαφές (calls): 19 φοιτητές
  Precision 47%: Από τους φοιτητές που καλέσαμε, το 47% πραγματικά αποχώρησε.
  Recall    14%: Εντοπίσαμε το 14% των συνολικών αποχωρήσεων· οι υπόλοιποι χάθηκαν.
  False Positive Rate 1.1%: Από τους φοιτητές που θα παρέμεναν, ανησυχήσαμε άδικα το 1.1%.
  Accuracy  93.0%
  Αν δεν καλούσαμε κανέναν, η ακρίβεια θα ήταν 93.1% (αλλά θα χάναμε το 100% των αποχωρήσεων).


*Your answer:*

✏️ **Your turn.** Now move the cut-off: make a small table for cut-offs 0.05, 0.1, 0.2, 0.3, 0.5 with the number contacted, the four boxes, precision and recall. What happens to the missed students and the false alarms as the cut-off goes down?

<details><summary>Hint</summary>

Write a function `boxes(cut)` that returns a dict with the counts and the rates, then `pd.DataFrame([boxes(c) for c in [0.05, 0.1, 0.2, 0.3, 0.5]])`.
</details>

In [47]:
cutoffs = [0.05, 0.1, 0.2, 0.3, 0.5]
rows = []
for c in cutoffs:
    res = boxes(c)
    total_calls = res["TP"] + res["FP"]
    precision = res["TP"] / total_calls if total_calls > 0 else 0
    recall = res["TP"] / (res["TP"] + res["FN"]) if (res["TP"] + res["FN"]) > 0 else 0
    rows.append({
        "cut-off": c,
        "contacted (calls)": total_calls,
        "TP (reached)": res["TP"],
        "FP (false alarm)": res["FP"],
        "FN (missed)": res["FN"],
        "TN (stayed)": res["TN"],
        "precision": f"{precision:.1%}",
        "recall": f"{recall:.1%}"
    })

pd.DataFrame(rows).set_index("cut-off")

,contacted (calls),TP (reached),FP (false alarm),FN (missed),TN (stayed),precision,recall
cut-off,,,,,,,
0.05,408,54,354,11,518,13.2%,83.1%
0.10,221,40,181,25,691,18.1%,61.5%
0.20,111,32,79,33,793,28.8%,49.2%
0.30,53,23,30,42,842,43.4%,35.4%
0.50,19,9,10,56,862,47.4%,13.8%


*Your answer:*

---
## 6 · A cut-off from costs 💶

Which cut-off is right depends on what each mistake costs. These are **assumptions**; part of your job is to
question them.

- **A conversation** costs about **500 DKK** of adviser time.
- **A false alarm** costs a worried student. Say that is worth **2,000 DKK**.
- **A student who leaves** costs the university about **60,000 DKK** (and the student far more). A conversation
  does not save everyone: say it keeps **30 %** of the students at risk.

So contacting a student with risk *p* is worth on average **p × 0.30 × 60,000 − 500 − (1 − p) × 2,000**.

In [48]:
COST_TALK, COST_WORRY, COST_LEAVE, HELPS = 500, 2_000, 60_000, 0.30

✏️ **Your turn.** Compute the net value of the rule for cut-offs from 0.02 to 0.9, plot it, and find the best cut-off. How many students would the office contact? Then change one assumption (a false alarm worth 10,000 DKK, or conversations that help only 10 %) and see how the rule moves. Who should decide these numbers? (Template: *A threshold from costs*, section 7 of the hotel notebook (session 10).)

<details><summary>Hint</summary>

Net value at a cut-off = TP × HELPS × COST_LEAVE − (TP + FP) × COST_TALK − FP × COST_WORRY. Break-even: the formula above is zero at p = (500 + 2,000) / (0.30 × 60,000 + 2,000).
</details>

In [49]:
COST_TALK, COST_WORRY, COST_LEAVE, HELPS = 500, 2_000, 60_000, 0.30

def net_value_total(y_true, y_prob, threshold):
    called = y_prob >= threshold
    tp = np.sum((called == 1) & (y_true == 1))
    fp = np.sum((called == 1) & (y_true == 0))
    return tp * HELPS * COST_LEAVE - (tp + fp) * COST_TALK - fp * COST_WORRY

thresholds = np.round(np.arange(0.05, 0.96, 0.01), 2)
values = [net_value_total(val["left"].values, p_val_final, t) for t in thresholds]
t_star = float(thresholds[int(np.argmax(values))])

p_break = COST_WORRY / (HELPS * COST_LEAVE + COST_WORRY)

print(f"best cut-off on validation: {t_star} (break-even would be {p_break:.3f})")
print(f"Max net value on validation: {max(values):,.0f} DKK")

best cut-off on validation: 0.24 (break-even would be 0.100)
Max net value on validation: 387,500 DKK


*Your answer:*

---
## 7 · Only 40 conversations

The office can talk to **40** students at week 6. With a fixed capacity the rule becomes: **talk to the 40 with the
highest risk**. How many of the 40 were really at risk (**precision at 40**)? How many of all students who left were
among them (**recall at 40**)?

✏️ **Your turn.** Compute precision and recall at 40. How many conversations would you need to reach half of the students who leave? What would a fourth adviser (about 13 more conversations) buy?

<details><summary>Hint</summary>

`top = val.nlargest(40, 'risk')`; precision = `top['left'].mean()`; recall = `top['left'].sum() / val['left'].sum()`. Try other numbers instead of 40.
</details>

In [50]:
top = val.nlargest(40, 'risk')
precision = top['left'].mean()
recall = top['left'].sum() / val['left'].sum()

print(f"Precision at 40: {precision:.1%}")
print(f"Recall at 40: {recall:.1%}")

Precision at 40: 52.5%
Recall at 40: 32.3%


In [51]:
top = val.nlargest(60, 'risk')
precision = top['left'].mean()
recall = top['left'].sum() / val['left'].sum()

print(f"Precision at 40: {precision:.1%}")
print(f"Recall at 40: {recall:.1%}")

Precision at 40: 43.3%
Recall at 40: 40.0%


While the how many calls we can make changes our accuracy overall drops but the ammount of students we save increases

*Your answer:*

---
## 8 · Fair to whom? ⚖️

About a fifth of the students are international. Compare the mistakes **per group** for the top-40 rule.

✏️ **Your turn.** For international and domestic students: the share who really leave, the average predicted risk, and the recall of the top-40 rule. Does the model get both groups right? Then compare how often each group logs in. Is a login the same signal for everyone? What would you tell the university?

<details><summary>Hint</summary>

Mark the top 40 first: `val['top40'] = val['risk'].rank(ascending=False) <= 40`. Then `val.groupby('international')[['left', 'risk']].mean()`; recall per group = among the students who left, the share with `top40`: `val[val['left'] == 1].groupby('international')['top40'].mean()`. Logins: `history.groupby('international')['logins_total'].mean()`.
</details>

In [52]:
val['top40'] = val['risk'].rank(ascending=False) <= 40
val.groupby('international')[['left', 'risk']].mean()
top40 =val[val['left'] == 1].groupby('international')['top40'].mean()
logins = history.groupby('international')['logins_total'].mean()
print(top40)
print(logins)

international
0    0.333333
1    0.285714
Name: top40, dtype: float64
international
0    36.446523
1    23.622296
Name: logins_total, dtype: float64


*Your answer:*

---
## 9 · Store the model

The app needs your model as files. Do it the way the hotel notebook (session 10) does in section 9, *Storing the model properly*: the
**portable copy** (`preprocess.json` + `booster.json`) loads in the app with any Python version, which a pickle does
not. The hotel app's `portable.py` does both the saving and the loading, for a pipeline built like the one in step 4.

✏️ **Your turn.** Save your XGBoost pipeline into a folder `model`, load it back with `portable.Model`, and check that it gives the same predictions. Download the folder as a zip. You will put it into your app's repository.

<details><summary>Hint</summary>

```python
import urllib.request, shutil
urllib.request.urlretrieve("https://raw.githubusercontent.com/aaubs/tonights-front-desk/main/portable.py", "portable.py")
import portable
os.makedirs("model", exist_ok=True)
portable.export(booster, "model")
back = portable.Model("model").predict_proba(val)
print(np.allclose(back, booster.predict_proba(val[features])[:, 1], atol=1e-5))
shutil.make_archive("model", "zip", "model")   # then download model.zip (Colab: the folder icon on the left)
```
</details>

In [53]:
import urllib.request, shutil, os
import numpy as np

urllib.request.urlretrieve("https://raw.githubusercontent.com/aaubs/tonights-front-desk/main/portable.py", "portable.py")
import portable

xgb_model = xgb.XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=3, subsample=0.8,
                               colsample_bytree=0.8, tree_method="hist", random_state=0)

xgb_pipe = make_pipeline(prepare, xgb_model)
xgb_pipe.fit(train[features], train["left"])
os.makedirs("model", exist_ok=True)
portable.export(xgb_pipe, "model")

back = portable.Model("model").predict_proba(val)
print("Are predictions identical?", np.allclose(back, xgb_pipe.predict_proba(val[features])[:, 1], atol=1e-5))

shutil.make_archive("model", "zip", "model")
print("model.zip created successfully!")

Are predictions identical? True
model.zip created successfully!


---
## 10 · This week's list

Now use the model the way the office would: on the 2026 students at the end of week 6.

✏️ **Your turn.** Predict the risk for every student in `new`, and show the 40 students the office should talk to this week. Would you send all 40 the same message? Is there anyone you would *not* contact?

<details><summary>Hint</summary>

`new['risk'] = booster.predict_proba(new[features])[:, 1]`, then `new.nlargest(40, 'risk')`.
</details>

In [54]:
new['risk'] = xgb_pipe.predict_proba(new[features])[:, 1]
new.nlargest(40, 'risk')

,student_id,cohort,age,admission_grade,international,first_gen,su_scholarship,fees_owed,moved_from_home,married,...,programme,gender,logins_total,logins_last3,logins_trend,submitted_share,missed_last3,quiz_mean,weeks_since_login,risk
336,S260391,2026,28,10.6,1,1,0,1,1,0,...,Communication & Digital Media,male,12,7.0,0.67,0.50,1.0,76.00,0.0,0.893550
234,S260271,2026,20,5.3,0,0,0,1,1,0,...,Communication & Digital Media,male,13,6.0,-0.33,0.33,3.0,37.00,1.0,0.857055
243,S260281,2026,38,4.3,0,1,0,0,1,1,...,Business Administration,female,21,8.0,-1.67,0.50,2.0,37.33,0.0,0.829652
566,S260648,2026,24,4.0,0,1,0,1,0,0,...,Tourism,male,11,3.0,-1.67,0.17,3.0,36.00,0.0,0.802798
461,S260529,2026,23,10.1,0,1,0,0,0,0,...,Business Administration,male,13,6.0,-0.33,0.50,2.0,57.33,0.0,0.756891
398,S260458,2026,20,5.6,0,1,0,1,1,0,...,Communication & Digital Media,male,8,0.0,-2.67,0.50,1.0,53.67,3.0,0.719939
81,S260095,2026,44,2.0,0,1,0,0,1,0,...,Sustainable Biotechnology,male,31,13.0,-1.67,0.50,1.0,39.00,0.0,0.717086
290,S260335,2026,32,10.2,0,0,0,0,0,1,...,Sustainable Biotechnology,male,43,16.0,-3.67,0.83,1.0,70.80,0.0,0.698709
193,S260223,2026,18,4.4,0,1,0,1,1,0,...,Social Work,female,17,7.0,-1.00,0.67,2.0,42.25,0.0,0.679969
525,S260602,2026,31,5.1,0,1,0,0,1,0,...,Sustainable Biotechnology,male,18,8.0,-0.67,0.67,1.0,53.50,0.0,0.662117


*Your answer:*

---
## 11 · Build the app 🛠️

Now build a Streamlit app for someone at the study office who is **not** a data scientist. Use the hotel app as your
template: [github.com/aaubs/tonights-front-desk](https://github.com/aaubs/tonights-front-desk) (`app.py`, `portable.py`,
`requirements.txt`, `model/`).

Your app should at least show:

1. **This week's list**: the 2026 students, ranked by risk, with the top 40 marked.
2. **The mistakes of a rule**: a slider (cut-off, or number of conversations) and, on the 2025 cohort, the four boxes
   in plain words ("12 students reached in time, 28 worried for nothing, 30 missed"), precision and recall.
3. **Per group**: the same numbers for international and domestic students.
4. **One thing of your own** that would help the office decide (for example the costs from step 6 as inputs, or why
   a student is on the list).

The app can read the data straight from GitHub, like this notebook (`URL + "new_week6.csv"`).

**Deploy:** create a public GitHub repository with `app.py`, `portable.py`, `requirements.txt` (`streamlit`,
`pandas`, `xgboost-cpu`, no version numbers) and your `model/` folder; then on
[share.streamlit.io](https://share.streamlit.io): *Create app* → your repository → `app.py`.